# Week 29, Day 1: __str__ and __repr__

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w029d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from datetime import date
d = date(2026, 11, 2)
words = ["it's", "ok"]
print(d)
print(repr(d))
print([d])
print(words, str(words[0]), repr(words[0]))

**Predict before you run:** what will this print?



In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Point({self.x!r}, {self.y!r})"

p = Point(3, 4)
print(p, [p, Point(0, -1)])
print(f"{p} and {p!r}", str(p) == repr(p))

In [ ]:
class Book:
    def __init__(self, title, author, year):
        self.title = title
        self.author = author
        self.year = year

    def __repr__(self):
        return f"{type(self).__name__}({self.title!r}, {self.author!r}, {self.year!r})"

shelf = [Book("Emma", "Austen", 1815), Book("1984", "Orwell", 1949)]
print(shelf)
again = eval(repr(shelf[0]))
print(again.title, again.year)

**Predict before you run:** what will this print?



In [ ]:
class Tag:
    def __init__(self, text):
        self.text = text

    def __repr__(self):
        return f"Tag({self.text})"

print([Tag("1"), Tag(1), Tag(" a ")])

In [ ]:
class Money:
    def __init__(self, pence):
        self.pence = pence

    def __repr__(self):
        return f"Money({self.pence})"

    def __str__(self):
        sign = "-" if self.pence < 0 else ""
        pounds, rest = divmod(abs(self.pence), 100)
        return f"{sign}£{pounds:,}.{rest:02d}"

prices = [Money(1250), Money(-350)]
print(prices[0], prices[1])
print(prices)
print(f"Total: {Money(900)}   (debug: {Money(900)!r})")

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    def __repr__(self):
        return f"Temperature({self.celsius!r})"

    def __str__(self):
        return f"{self.celsius:.1f}°C"

    def __format__(self, spec):
        if spec == "f":
            return f"{self.celsius * 9 / 5 + 32:.1f}°F"
        if spec in ("", "c"):
            return str(self)
        return format(str(self), spec)

t = Temperature(21.5)
print(f"{t}  {t:f}  {t:c}  [{t:>10}]")

> **Key idea.** `str(x)` (and `print`) is for users; `repr(x)` is for programmers, and is what containers, the prompt and `!r` show. Always write `__repr__`, returning a string that shows exactly what the object is, ideally as code that recreates it, with `!r` on each attribute. Add `__str__` for a friendlier form, and `__format__` to support your own format codes.

### Exercise: A useful repr

Add `__repr__` to `Vector` so that `repr(Vector(1.5, -2))` is `Vector(1.5, -2)`, and `eval(repr(v))` gives a vector with the same `x` and `y`. Use `type(self).__name__` and `!r`.

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

print([Vector(1.5, -2), Vector(0, 0)])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"repr\"\n    expect_output(\"[Vector(1.5, -2), Vector(0, 0)]\")\n    V = ns[\"Vector\"]\n    v = eval(repr(V(3, 4)), {\"Vector\": V})\n    assert (v.x, v.y) == (3, 4) and \"type(self).__name__\" in source")

### Exercise: For programmers and for people

Give `Contact` a `__repr__` like `Contact('Ada Obi', 'ada@x.ng')` and a `__str__` like `Ada Obi <ada@x.ng>`. Then print one contact, and the list `people`, and then the people one per line in their friendly form.

In [ ]:
class Contact:
    def __init__(self, name, email):
        self.name = name
        self.email = email

people = [Contact("Ada Obi", "ada@x.ng"), Contact("Bo Musa", "bo@y.com")]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"three kinds of output\"\n    expect_output(\"Ada Obi <ada@x.ng>\\n[Contact('Ada Obi', 'ada@x.ng'), Contact('Bo Musa', 'bo@y.com')]\\nAda Obi <ada@x.ng>\\nBo Musa <bo@y.com>\")")

### Exercise: When code won't do

A `Playlist` may hold hundreds of songs, so its repr shouldn't list them all. Write `__repr__` returning `<Playlist 'Road trip': 3 songs, 11:05>` (the total duration as minutes and two-digit seconds; `1 song` for one), and `__str__` returning the name and then each song on its own line, numbered, like `1. Hey Jude (7:11)`.

In [ ]:
class Playlist:
    def __init__(self, name):
        self.name = name
        self.songs = []

    def add(self, title, seconds):
        self.songs.append((title, seconds))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"repr and str\"\n    P = ns[\"Playlist\"]\n    p = P(\"Road trip\")\n    p.add(\"Hey Jude\", 431); p.add(\"Golden\", 205); p.add(\"Africa\", 29)\n    assert repr(p) == \"<Playlist 'Road trip': 3 songs, 11:05>\"\n    assert str(p) == \"Road trip\\n1. Hey Jude (7:11)\\n2. Golden (3:25)\\n3. Africa (0:29)\"\n    q = P(\"One\")\n    q.add(\"Solo\", 60)\n    assert repr(q) == \"<Playlist 'One': 1 song, 1:00>\" and repr(P(\"Empty\")) == \"<Playlist 'Empty': 0 songs, 0:00>\"")

### Exercise: Your own format codes

Give `Distance` (stored in metres) a `__format__` supporting `km` (`"5.20 km"`, two places), `mi` (miles, 1609.344 m each, two places: `"3.23 mi"`), and `m` or an empty spec (`"5200 m"`, rounded to a whole number). Any other spec should apply to the metres text, so `f"{d:>10}"` right-aligns `"5200 m"`.

In [ ]:
class Distance:
    def __init__(self, metres):
        self.metres = metres

    def __str__(self):
        return f"{round(self.metres)} m"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"formats\"\n    D = ns[\"Distance\"]\n    d = D(5200)\n    assert f\"{d:km}|{d:mi}|{d:m}|{d}|{d:>10}|\" == \"5.20 km|3.23 mi|5200 m|5200 m|    5200 m|\"\n    assert f\"{D(999.6)}\" == \"1000 m\" and f\"{D(1609.344):mi}\" == \"1.00 mi\"")

### Bug hunt: It printed, then crashed

Printing a `Card` shows `Q♥` on one line and then crashes with `TypeError: __str__ returned non-string (type NoneType)`. Fix `__str__`.

In [ ]:
class Card:
    def __init__(self, rank, suit):
        self.rank = rank
        self.suit = suit

    def __str__(self):
        print(f"{self.rank}{self.suit}")

print(Card("Q", "♥"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"prints once\"\n    expect_output(\"Q\u2665\")")

### Bug hunt: Which is which?

These two items look the same in the list, but one has the number 7 and the other the text `"7"`, and a third has a space at the end of its name. Fix `__repr__` so they're all told apart.

In [ ]:
class Item:
    def __init__(self, name, code):
        self.name = name
        self.code = code

    def __repr__(self):
        return f"Item({self.name}, {self.code})"

print([Item("pen", 7), Item("pen", "7"), Item("pen ", 7)])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"unambiguous\"\n    expect_output(\"[Item('pen', 7), Item('pen', '7'), Item('pen ', 7)]\")")

### Puzzle: Repr and str

Arrange the lines into a `Score` class with both methods. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
class A:
    def __repr__(self):
        return "A()"

class B:
    def __repr__(self):
        return "B()"
    def __str__(self):
        return "a B"

p = ___   # str(A())
q = ___   # str(B())
r = ___   # str([B()])
s = ___   # f"{B()!r} {B()}"
t = ___   # repr("it's")  (write the exact text you'd see)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r, s, t) == (\"A()\", \"a B\", \"[B()]\", \"B() a B\", \"\\\"it's\\\"\")")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a printable grid

Write `Grid(rows)` for a rectangular grid of numbers (a list of equal-length lists; copy it; `ValueError` if the rows differ in length or there are none). `__repr__` gives code that recreates it: `Grid([[1, 2], [3, 40]])`. `__str__` lays it out with each column right-aligned to its widest entry, columns separated by two spaces, and a border. For `Grid([[1, 2], [3, 40]])`:

```
+-------+
| 1   2 |
| 3  40 |
+-------+
```

Precisely: each row line is `"| "` + the cells joined by two spaces + `" |"`, and the top and bottom border is `"+"` + `"-"` × (the row line's length − 2) + `"+"`. Numbers are shown with `str`. `__format__` with spec `"compact"` gives the rows separated by `/` and cells by `,`: `1,2/3,40`.

In [ ]:
class Grid:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_repr():\n    \"repr\"\n    G = ns[\"Grid\"]\n    g = G([[1, 2], [3, 40]])\n    assert repr(g) == \"Grid([[1, 2], [3, 40]])\"\n    rows = [[1, 2]]\n    g2 = G(rows)\n    rows[0][0] = 99\n    assert repr(g2) == \"Grid([[1, 2]])\", \"copy the rows (each inner list too)\"\ndef test_str():\n    \"str\"\n    G = ns[\"Grid\"]\n    assert str(G([[1, 2], [3, 40]])) == \"+-------+\\n| 1   2 |\\n| 3  40 |\\n+-------+\"\ndef test_layout_exact():\n    \"exact layout\"\n    G = ns[\"Grid\"]\n    s = str(G([[-1.5, 10, 7], [200, 3, 0]])).split(\"\\n\")\n    assert s[1] == \"| -1.5  10  7 |\" and s[2] == \"|  200   3  0 |\" and s[0] == \"+\" + \"-\" * (len(s[1]) - 2) + \"+\" and s[3] == s[0]\ndef test_format_errors():\n    \"compact format, and errors\"\n    G = ns[\"Grid\"]\n    assert f\"{G([[1, 2], [3, 40]]):compact}\" == \"1,2/3,40\"\n    for bad in ([], [[1, 2], [3]]):\n        try:\n            G(bad)\n            assert False, f\"Grid({bad}) should raise ValueError\"\n        except ValueError:\n            pass")

## Warm-up

The same date prints two ways. `print(d)` gives the friendly `2026-11-02`; `repr(d)` gives `datetime.date(2026, 11, 2)`, which says exactly what the object is, and is even the code that would make it. A list prints its items with `repr`, which is why the strings inside it show their quotes.

## Two kinds of text

Every object can be turned into text in two ways:

| | `str(x)` | `repr(x)` |
|---|---|---|
| Meant for | users, in output | programmers, when debugging |
| Should be | readable and friendly | unambiguous: say exactly what the object is |
| Used by | `print(x)`, `f"{x}"`, `str(x)` | the interactive prompt, lists and dictionaries showing their contents, `f"{x!r}"`, error messages, the debugger |
| Special method | `__str__` | `__repr__` |

For many types the two are the same (`str(42)` and `repr(42)` are both `42`). For strings they differ: `str("ok")` is `ok`, but `repr("ok")` is `'ok'`, with quotes, so you can tell the string `"42"` from the number `42`, and see spaces at the ends.

If a class defines only `__repr__`, `str()` uses it too. If it defines neither, both give the unhelpful default. So **always write `__repr__` first**; add `__str__` only when users need a different, friendlier form.

## Writing __repr__

A good `__repr__`:

- returns a **string** (it must; returning anything else raises `TypeError`);
- names the class and shows the attributes that matter;
- where reasonable, looks like the **code that would recreate the object**: `Point(3, 4)`, so that copying it from the output into Python gives an equal object;
- uses **`!r`** for each attribute, so strings keep their quotes and nested objects show their own repr.

`type(self).__name__` instead of writing `Book` keeps the repr right for subclasses (Week 30). `eval` turns text back into a value by running it as Python, which shows the repr really is recreating code. (Never `eval` text from users or files: it runs whatever is there.)

When an object can't sensibly be written as code (it holds an open file, or hundreds of items), use the angle-bracket style Python uses for such things, with the most useful details: `<Basket 3 items, £12.40>`.

Without `!r`, strings lose their quotes and the repr becomes ambiguous:

## Writing __str__

`__str__` is for the person using your program: whatever reads best in output, with no need to show every detail.

Notice `print(prices)` uses the **repr** of each item, even though `Money` has a `__str__`: containers always show their contents' repr. To print a list of friendly forms, join them yourself: `", ".join(str(m) for m in prices)`.

## Formatting with __format__

Format specs like `{x:>10}` and `{x:.2f}` call the object's `__format__(spec)` method. Python's types read the spec themselves; your class can too, to support its own codes. A simple and common pattern is to handle your own codes and pass anything else on:

`format(value, spec)` is what an f-string calls for you; passing `str(self)` on lets ordinary width and alignment specs still work.

## Practice

## Quiz

## Daily challenge

## Before you go

Add a `__repr__` to every class you wrote last week, and print a list of their instances. Tomorrow: equality and hashing, so `Point(1, 2) == Point(1, 2)` and points can go in sets.